# Marathi + Gujarati translation: NLLB vs IndicTrans2 (zero-shot, LoRA)

Experiment log, **cleaned**. Run it **stage by stage, restarting the kernel between stages** (NLLB, IndicTrans2 eval, LoRA training, LoRA eval). Loading several models in one kernel pushed a 16GB Mac into swap and made MPS 10x slower.

Every stage starts with the same **Setup** cell. Results are written to `results/` after each direction, so a crashed run resumes where it stopped.

Before you start:
- `pip install torch transformers==4.46.3 "datasets<4" peft accelerate sacrebleu sentencepiece pandas tqdm gTTS cython setuptools`, plus `IndicTransToolkit` from GitHub.
- Hugging Face: IN22-Gen is gated, and IndicTrans2 repos needed accepted terms. Run `huggingface-cli login` in a terminal. **Do not paste tokens or API keys into this notebook.**
- Not done here: IN22 evaluation (loaded only), significance tests.

## Setup (run at the start of every stage)

In [ ]:
import os, time, random, glob, json, re, gc, tarfile, urllib.request
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch, pandas as pd, sacrebleu
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from IndicTransToolkit.processor import IndicProcessor
from datasets import load_dataset

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
for d in ["data", "results", "results/hyps", "adapters"]:
    os.makedirs(d, exist_ok=True)

ENG, HIN, MAR, GUJ = "eng_Latn", "hin_Deva", "mar_Deva", "guj_Gujr"
LANG_CODES = [ENG, HIN, MAR, GUJ]
PAIRS = [(ENG, MAR), (MAR, ENG), (ENG, GUJ), (GUJ, ENG),
         (HIN, MAR), (MAR, HIN), (HIN, GUJ), (GUJ, HIN),
         (MAR, GUJ), (GUJ, MAR)]
MG = [(MAR, GUJ), (GUJ, MAR)]
ENG_PAIRS = [p for p in PAIRS if ENG in p]

# FLORES-200 devtest (1012 sentences)
if not os.path.exists("data/flores200_dataset"):
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz",
                               "data/flores200.tar.gz")
    with tarfile.open("data/flores200.tar.gz") as t:
        t.extractall("data")

def load_flores(lang, split="devtest"):
    with open(f"data/flores200_dataset/{split}/{lang}.{split}", encoding="utf-8") as f:
        return [l.strip() for l in f]

FLORES = {l: load_flores(l) for l in LANG_CODES}

# IN22-Gen (gated: needs HF login + accepted terms). Columns are the FLORES language codes.
in22 = load_dataset("ai4bharat/IN22-Gen", trust_remote_code=True)
ds = in22[list(in22.keys())[0]]
IN22 = {l: list(ds[l]) for l in LANG_CODES}
TESTSETS = {"flores": FLORES, "in22": IN22}

def batched(fn, texts, src, tgt, bs=8):
    # sort by length: less padding, faster
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    out = [None] * len(texts)
    for i in tqdm(range(0, len(order), bs), desc=f"{src}->{tgt}", leave=False):
        idx = order[i:i+bs]
        res = fn([texts[j] for j in idx], src, tgt)
        for j, r in zip(idx, res):
            out[j] = r
    return out

def score(hyps, refs):
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score   # chrF++
    return round(bleu, 2), round(chrf, 2)

def evaluate(fn, system_name, testset_name, n=None, pairs=PAIRS):
    # full test set (n=None). Saves a CSV row after every direction and skips finished ones.
    ts = TESTSETS[testset_name]
    path = f"results/{system_name}_{testset_name}.csv"
    rows = pd.read_csv(path).to_dict("records") if os.path.exists(path) else []
    done_dirs = {r["direction"] for r in rows}
    for s, t in pairs:
        d = f"{s[:3]}->{t[:3]}"
        if d in done_dirs:
            print("skip (already done):", d); continue
        src, ref = ts[s][:n], ts[t][:n]
        t0 = time.time()
        hyp = batched(fn, src, s, t)
        b, c = score(hyp, ref)
        rows.append(dict(system=system_name, testset=testset_name, direction=d,
                         BLEU=b, chrF2pp=c, secs=round(time.time()-t0)))
        print(rows[-1])
        pd.DataFrame(rows).to_csv(path, index=False)
    return pd.DataFrame(rows)

# Same, but on the first 100 FLORES sentences, in the format used to merge with the main tables
N_EVAL = 100
K_CSV = "results/flores100_gu_mr.csv"

def evaluate_k(fn, model_name, pairs, n=N_EVAL):
    rows = pd.read_csv(K_CSV).to_dict("records") if os.path.exists(K_CSV) else []
    done = {(r["model"], r["src"], r["tgt"], r["n"]) for r in rows}
    for s, t in pairs:
        if (model_name, s, t, n) in done:
            print("skip:", model_name, s, t); continue
        src, ref = FLORES[s][:n], FLORES[t][:n]
        t0 = time.time()
        hyp = batched(fn, src, s, t)
        b, c = score(hyp, ref)
        json.dump(hyp, open(f"results/hyps/{model_name}_{s}_{t}.json", "w"), ensure_ascii=False)
        rows.append({"model": model_name, "src": s, "tgt": t, "n": n, "bleu": b, "chrf++": c})
        pd.DataFrame(rows).to_csv(K_CSV, index=False)
        print(rows[-1], "|", round(time.time() - t0), "s")
    return pd.DataFrame(rows)

def make_pivot_fn(base_fn, pivot):
    def fn(texts, src, tgt):
        mid = base_fn(texts, src, pivot)
        return base_fn(mid, pivot, tgt)
    return fn

print("device:", device, "| flores:", len(FLORES[MAR]), "| in22:", len(IN22[MAR]))

## Stage A: NLLB baseline (restart kernel first, then Setup)

In [ ]:
nllb_name = "facebook/nllb-200-distilled-600M"
nllb_tok = AutoTokenizer.from_pretrained(nllb_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_name).to(device).eval()

@torch.no_grad()
def nllb_translate(texts, src, tgt):
    nllb_tok.src_lang = src
    inputs = nllb_tok(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=256).to(device)
    out = nllb_model.generate(**inputs,
        forced_bos_token_id=nllb_tok.convert_tokens_to_ids(tgt),
        num_beams=5, max_new_tokens=256)
    return nllb_tok.batch_decode(out, skip_special_tokens=True)

print(nllb_translate(["How are you?"], ENG, MAR), nllb_translate(["How are you?"], ENG, GUJ))

In [ ]:
# slow (about 15-25 min per direction on an M-series Mac)
NLLB_PAIRS = [(ENG, MAR), (MAR, ENG), (ENG, GUJ), (GUJ, ENG), (MAR, GUJ), (GUJ, MAR)]
evaluate(nllb_translate, "nllb", "flores", pairs=NLLB_PAIRS)

In [ ]:
# first 100 sentences (a few minutes per direction)
evaluate_k(nllb_translate, "nllb", NLLB_PAIRS)

## Stage B: IndicTrans2 zero-shot and pivots (restart kernel first, then Setup)

In [ ]:
ip = IndicProcessor(inference=True)

IT2_MODELS = {
    "en-indic":    "ai4bharat/indictrans2-en-indic-dist-200M",
    "indic-en":    "ai4bharat/indictrans2-indic-en-dist-200M",
    "indic-indic": "ai4bharat/indictrans2-indic-indic-dist-320M",
}

def direction_of(src, tgt):
    if src == ENG: return "en-indic"
    if tgt == ENG: return "indic-en"
    return "indic-indic"

_cache = {}
def load_it2(direction):
    if direction not in _cache:
        name = IT2_MODELS[direction]
        tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(name, trust_remote_code=True).to(device).eval()
        _cache[direction] = (tok, mdl)
    return _cache[direction]

@torch.no_grad()
def _it2_generate(tok, mdl, texts, src, tgt):
    batch = ip.preprocess_batch(texts, src_lang=src, tgt_lang=tgt)
    inputs = tok(batch, truncation=True, padding="longest", max_length=256,
                 return_tensors="pt", return_attention_mask=True).to(device)
    out = mdl.generate(**inputs, num_beams=5, max_length=256, num_return_sequences=1)
    dec = tok.batch_decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return ip.postprocess_batch(dec, lang=tgt)

def it2_translate(texts, src, tgt):
    tok, mdl = load_it2(direction_of(src, tgt))
    return _it2_generate(tok, mdl, texts, src, tgt)

print(it2_translate(["How are you?"], ENG, MAR), it2_translate(["तुम्ही कुठे जात आहात?"], MAR, GUJ))

In [ ]:
# full FLORES: zero-shot (direct mar<->guj is included in MG)
evaluate(it2_translate, "it2_zeroshot", "flores", pairs=[(ENG, MAR), (MAR, ENG), (ENG, GUJ), (GUJ, ENG)] + MG)

In [ ]:
# full FLORES: Marathi <-> Gujarati via Hindi and via English
it2_pivot_hin = make_pivot_fn(it2_translate, HIN)
it2_pivot_eng = make_pivot_fn(it2_translate, ENG)
evaluate(it2_pivot_hin, "it2_pivot_hin", "flores", pairs=MG)
evaluate(it2_pivot_eng, "it2_pivot_eng", "flores", pairs=MG)

In [ ]:
# first 100 sentences
evaluate_k(it2_translate, "it2_zeroshot", [(ENG, MAR), (MAR, ENG), (ENG, GUJ), (GUJ, ENG), (MAR, GUJ), (GUJ, MAR)])
evaluate_k(it2_translate, "it2_zeroshot", [(HIN, MAR), (MAR, HIN), (HIN, GUJ), (GUJ, HIN)])
evaluate_k(it2_pivot_hin, "it2_pivot_hin", MG)
evaluate_k(it2_pivot_eng, "it2_pivot_eng", MG)

## Stage C: LoRA training data and training (restart kernel first, then Setup)

Training data: Samanantar (`ai4bharat/samanantar`, configs `mr` and `gu`, `src` = English, `tgt` = Indic), 20k filtered pairs per language. FLORES and IN22 sentences are removed so there is no test leakage.

**Important lesson:** the Indic side of every training pair must also go through `IndicProcessor` (`prep_target` below). With raw Gujarati targets the loss was about 10.1, with processed targets about 2.3. The first training attempt used raw targets and was thrown away.

In [ ]:
ip = IndicProcessor(inference=True)

IT2_MODELS = {
    "en-indic":    "ai4bharat/indictrans2-en-indic-dist-200M",
    "indic-en":    "ai4bharat/indictrans2-indic-en-dist-200M",
    "indic-indic": "ai4bharat/indictrans2-indic-indic-dist-320M",
}

N_PER_LANG = 20000
banned = set()
for ts in TESTSETS.values():
    for l, sents in ts.items():
        banned.update(s.strip() for s in sents)

SCRIPT_RANGE = {"mr": (0x0900, 0x097F), "gu": (0x0A80, 0x0AFF)}

def script_ratio(text, lang):
    lo, hi = SCRIPT_RANGE[lang]
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(lo <= ord(c) <= hi for c in letters) / len(letters)

def good_pair(en, ind, lang):
    if en in banned or ind in banned:
        return False
    en_w, in_w = len(en.split()), len(ind.split())
    if not (4 <= en_w <= 60 and 4 <= in_w <= 60):
        return False
    if not (0.5 <= in_w / en_w <= 2.0):
        return False
    if script_ratio(ind, lang) < 0.7:
        return False
    if re.search(r"[\[\]<>{}]|http|www\.", en + ind):
        return False
    if sum(c.isdigit() for c in ind) > 0.3 * len(ind):
        return False
    return True

def get_pairs(cfg, n):
    ds = load_dataset("ai4bharat/samanantar", cfg, split="train", streaming=True, trust_remote_code=True)
    ds = ds.shuffle(seed=42, buffer_size=20000)
    pairs, seen, scanned = [], set(), 0
    for ex in ds:
        scanned += 1
        en, ind = ex["src"].strip(), ex["tgt"].strip()
        if en in seen or not good_pair(en, ind, cfg):
            continue
        seen.add(en)
        pairs.append((en, ind))
        if len(pairs) >= n:
            break
    print(f"{cfg}: kept {len(pairs)} of {scanned} scanned")
    return pairs

if os.path.exists("data/train_pairs.json"):
    DATA = {k: [tuple(p) for p in v] for k, v in json.load(open("data/train_pairs.json")).items()}
else:
    DATA = {MAR: get_pairs("mr", N_PER_LANG), GUJ: get_pairs("gu", N_PER_LANG)}
    json.dump(DATA, open("data/train_pairs.json", "w"), ensure_ascii=False)
print({k: len(v) for k, v in DATA.items()})

In [ ]:
from peft import LoraConfig, get_peft_model

def prep_target(texts, lang):
    # IndicProcessor prepends "src_lang tgt_lang " tags; drop them and keep the processed text
    out = ip.preprocess_batch(texts, src_lang=lang, tgt_lang=ENG)
    return [o.split(" ", 2)[2] for o in out]

def train_lora(direction, epochs=1, bs=4, lr=2e-4, max_len=128, save_dir=None):
    save_dir = save_dir or f"adapters/lora_{direction}"
    name = IT2_MODELS[direction]
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    base = AutoModelForSeq2SeqLM.from_pretrained(name, trust_remote_code=True)

    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                     target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"])
    model = get_peft_model(base, cfg).to(device)
    model.print_trainable_parameters()   # about 3%

    examples = []
    for lang, pairs in DATA.items():
        en = [p[0] for p in pairs]; ind = [p[1] for p in pairs]
        if direction == "en-indic":
            src = ip.preprocess_batch(en, src_lang=ENG, tgt_lang=lang)
            tgt = prep_target(ind, lang)
        else:
            src = ip.preprocess_batch(ind, src_lang=lang, tgt_lang=ENG)
            tgt = prep_target(en, ENG)
        examples += list(zip(src, tgt))
    random.seed(0); random.shuffle(examples)

    def encode(batch):
        enc = tok([b[0] for b in batch], text_target=[b[1] for b in batch],
                  truncation=True, padding="longest", max_length=max_len, return_tensors="pt")
        lab = enc["labels"].clone(); lab[lab == tok.pad_token_id] = -100
        return enc["input_ids"].to(device), enc["attention_mask"].to(device), lab.to(device)

    # sanity check before training: base loss should be about 1-4. If it is 7+, stop.
    ids, am, lab = encode(examples[:16])
    model.eval()
    with torch.no_grad():
        l0 = model(input_ids=ids, attention_mask=am, labels=lab).loss.item()
    print("SRC  :", examples[0][0][:100]); print("TGT  :", examples[0][1][:100])
    print(f"BASE LOSS before training: {l0:.3f}")
    model.train()

    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    steps_total = epochs * (len(examples) // bs)
    sched = torch.optim.lr_scheduler.LinearLR(opt, 1.0, 0.1, total_iters=steps_total)
    step, t_start, run = 0, time.time(), []

    for ep in range(epochs):
        random.shuffle(examples)
        for i in range(0, len(examples) - bs + 1, bs):
            ids, am, lab = encode(examples[i:i+bs])
            out = model(input_ids=ids, attention_mask=am, labels=lab)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            step += 1; run.append(out.loss.item())
            if step % 500 == 0:
                print(f"ep{ep} step {step}/{steps_total} avg-loss(50) {sum(run[-50:])/50:.3f} "
                      f"| {(time.time()-t_start)/60:.1f} min")
                model.save_pretrained(save_dir)
    model.save_pretrained(save_dir)
    print("saved:", save_dir, "| total", round((time.time()-t_start)/60, 1), "min")
    del model, base
    gc.collect()
    if device == "mps": torch.mps.empty_cache()

In [ ]:
# quick trial first (about 1 minute): check that BASE LOSS is about 1-4 for BOTH directions
DATA_FULL = DATA
DATA = {k: v[:400] for k, v in DATA_FULL.items()}
train_lora("en-indic", bs=4, save_dir="adapters/trial_en_indic")
train_lora("indic-en", bs=4, save_dir="adapters/trial_indic_en")
DATA = DATA_FULL

In [ ]:
# real training: 10,000 steps each (about 50 min and 40 min on an M-series Mac). Keep the laptop plugged in.
train_lora("en-indic", epochs=1, bs=4)

In [ ]:
# restart the kernel here, run Setup + the first cell of Stage C (data), then:
# train_lora("indic-en", epochs=1, bs=4)

## Stage D: LoRA evaluation (restart kernel first, then Setup)

Result: LoRA ended **below** zero-shot on all four English directions (see the final table). We did not test why.

In [ ]:
ip = IndicProcessor(inference=True)

IT2_MODELS = {
    "en-indic":    "ai4bharat/indictrans2-en-indic-dist-200M",
    "indic-en":    "ai4bharat/indictrans2-indic-en-dist-200M",
    "indic-indic": "ai4bharat/indictrans2-indic-indic-dist-320M",
}

def direction_of(src, tgt):
    if src == ENG: return "en-indic"
    if tgt == ENG: return "indic-en"
    return "indic-indic"

from peft import PeftModel

@torch.no_grad()
def _it2_generate(tok, mdl, texts, src, tgt):
    batch = ip.preprocess_batch(texts, src_lang=src, tgt_lang=tgt)
    inputs = tok(batch, truncation=True, padding="longest", max_length=256,
                 return_tensors="pt", return_attention_mask=True).to(device)
    out = mdl.generate(**inputs, num_beams=5, max_length=256, num_return_sequences=1)
    dec = tok.batch_decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return ip.postprocess_batch(dec, lang=tgt)

_lora_cache = {}
def load_it2_lora(direction):
    if direction not in _lora_cache:
        name = IT2_MODELS[direction]
        tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        base = AutoModelForSeq2SeqLM.from_pretrained(name, trust_remote_code=True)
        m = PeftModel.from_pretrained(base, f"adapters/lora_{direction}")
        m = m.merge_and_unload().to(device).eval()   # a "tie_word_embeddings" warning here was harmless
        _lora_cache[direction] = (tok, m)
    return _lora_cache[direction]

def it2_lora_translate(texts, src, tgt):
    d = direction_of(src, tgt)
    if d == "indic-indic":
        raise ValueError("no LoRA adapter for indic-indic; use the English pivot")
    tok, mdl = load_it2_lora(d)
    return _it2_generate(tok, mdl, texts, src, tgt)

print(it2_lora_translate(["How are you?"], ENG, MAR), it2_lora_translate(["तुम्ही कसे आहात?"], MAR, ENG))

In [ ]:
# full FLORES
evaluate(it2_lora_translate, "it2_lora", "flores", pairs=[(ENG, MAR), (MAR, ENG), (ENG, GUJ), (GUJ, ENG)])
it2_lora_pivot_eng = make_pivot_fn(it2_lora_translate, ENG)
evaluate(it2_lora_pivot_eng, "it2_lora_pivot_eng", "flores", pairs=MG)

In [ ]:
# first 100 sentences
evaluate_k(it2_lora_translate, "it2_lora", [(ENG, MAR), (MAR, ENG), (ENG, GUJ), (GUJ, ENG)])
evaluate_k(it2_lora_pivot_eng, "it2_lora_pivot_eng", MG)

## Final tables (FLORES)

In [ ]:
all_df = pd.concat([pd.read_csv(f) for f in glob.glob("results/*_flores.csv") if "test" not in f])
all_df = all_df[all_df.system != "it2_direct"]
all_df = all_df.drop_duplicates(["system", "testset", "direction"])

main = all_df.pivot_table(index="direction", columns="system", values=["BLEU", "chrF2pp"])
main.to_csv("results/final_table_flores.csv")
print(main.round(2).to_string())

mg = all_df[all_df.direction.isin(["mar->guj", "guj->mar"])]
print(mg.pivot_table(index="system", columns="direction", values=["BLEU", "chrF2pp"]).round(2).to_string())

## Error analysis (8 random FLORES sentences, English to Marathi/Gujarati)

Restart kernel, run Setup and the Stage D model cell first. Eight sentences are only examples, not evidence of a pattern.

In [ ]:
zs_tok = AutoTokenizer.from_pretrained(IT2_MODELS["en-indic"], trust_remote_code=True)
zs_mdl = AutoModelForSeq2SeqLM.from_pretrained(IT2_MODELS["en-indic"], trust_remote_code=True).to(device).eval()

def zs_en_indic(texts, src, tgt):
    return _it2_generate(zs_tok, zs_mdl, texts, src, tgt)

random.seed(1)
idx = random.sample(range(len(FLORES[ENG])), 8)
src = [FLORES[ENG][i] for i in idx]
for tgt in [MAR, GUJ]:
    ref = [FLORES[tgt][i] for i in idx]
    zs = batched(zs_en_indic, src, ENG, tgt)
    lo = batched(it2_lora_translate, src, ENG, tgt)
    print("=" * 30, tgt, "=" * 30)
    for s, r, a, b in zip(src, ref, zs, lo):
        print("SRC :", s, "\nREF :", r, "\nZERO:", a, "\nLORA:", b, "\n" + "-" * 60)

## Voice: ASR and TTS (separate session, no translation models loaded)

ASR: Whisper `small` was tried first and was poor on Marathi/Gujarati (average WER 0.75 and 1.0 on 3 synthetic gTTS sentences each, a tiny sample). The app uses **Sarvam Saaras v3** (cloud API, so audio leaves the laptop). We did not measure its WER.

Needs `pip install sarvamai`, ffmpeg on PATH, and your own key. The key is read with `getpass`, never stored in the notebook.

In [ ]:
import subprocess, shutil
from getpass import getpass
from sarvamai import SarvamAI

print("ffmpeg:", shutil.which("ffmpeg"))
os.environ["SARVAM_API_KEY"] = getpass("Sarvam API key: ")
sarvam = SarvamAI(api_subscription_key=os.environ["SARVAM_API_KEY"])
SARVAM_LANG = {"mr": "mr-IN", "gu": "gu-IN", "hi": "hi-IN", "en": "en-IN"}

def to_wav(path):
    out = os.path.splitext(path)[0] + ".wav"
    subprocess.run(["ffmpeg", "-y", "-i", path, "-ar", "16000", "-ac", "1", out],
                   check=True, capture_output=True)
    return out

def transcribe(audio_path, lang="mr"):          # lang: "mr", "gu", "hi", "en"
    wav = audio_path if audio_path.endswith(".wav") else to_wav(audio_path)
    with open(wav, "rb") as f:
        resp = sarvam.speech_to_text.transcribe(file=f, model="saaras:v3",
                                                language_code=SARVAM_LANG[lang], mode="transcribe")
    return resp.transcript.strip()

In [ ]:
from gtts import gTTS

def speak(text, lang="mr", out="out.mp3"):      # needs internet
    gTTS(text=text, lang=lang).save(out)
    return out

TESTS = {"mr": ["नमस्कार, तुम्ही कसे आहात?", "रेल्वे स्टेशन कुठे आहे?", "मला पाणी हवे आहे."],
         "gu": ["નમસ્તે, તમે કેમ છો?", "રેલ્વે સ્ટેશન ક્યાં આવેલું છે?", "મને પાણી જોઈએ છે."]}

def wer(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1] + (r[i-1] != h[j-1]))
    return d[len(r)][len(h)] / max(len(r), 1)

def norm(t):
    return re.sub(r"[^\w\s]", "", t).strip()

rows = []
for lang, sents in TESTS.items():
    for i, s in enumerate(sents):
        f = speak(s, lang, f"test_{lang}_{i}.mp3")
        hyp = transcribe(f, lang)
        w = wer(norm(s), norm(hyp))
        rows.append((lang, w))
        print(f"[{lang}] REF: {s}\n     HYP: {hyp}\n     WER: {w:.2f}\n")
for lang in ["mr", "gu"]:
    ws = [w for l, w in rows if l == lang]
    print(lang, "average WER (synthetic speech, 3 sentences):", round(sum(ws)/len(ws), 2))

## Using the result in the app

Copy `translate_gu_mr.py` and the `adapters/` folder (only needed if `USE_LORA = True`). Then:

```python
from translate_gu_mr import translate
translate("Where is the railway station?", "eng_Latn", "mar_Deva")
```